# 💎 Mirror Nexus Bot — Heroku Deployment (FIXED)

**Developer:** [@Venuboyy](https://t.me/Venuboyy)

This notebook will deploy your Mirror Nexus Bot to Heroku step by step. **Now with automatic authentication fix.**

---

## Prerequisites
- GitHub repository with your bot code
- Heroku account
- MongoDB Atlas URI
- Bot Token from [@BotFather](https://t.me/BotFather)
- API ID & Hash from [my.telegram.org](https://my.telegram.org)

---

## Step 1: Install Heroku CLI

In [1]:
# Install Heroku CLI
!curl https://cli-assets.heroku.com/install.sh | sh
!heroku --version

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  2034  100  2034    0     0  11771      0 --:--:-- --:--:-- --:--:-- 11757
Installing CLI from https://cli-assets.heroku.com/channels/stable/heroku-linux-x64.tar.xz
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 55.5M  100 55.5M    0     0  3951k      0  0:00:14  0:00:14 --:--:-- 7254k
v22.22.2
heroku installed to /usr/local/bin/heroku
 ›   Warning: Our terms of service have changed: 
 ›   https://dashboard.heroku.com/terms-of-service
heroku/11.3.0 linux-x64 node-v22.22.2
heroku/11.3.0 linux-x64 node-v22.22.2


## Step 2: Login to Heroku (Auto-Auth Fix)

You need your Heroku API key. Get it from: https://dashboard.heroku.com/account

Scroll down to **API Key** → click **Reveal** → copy it.

In [2]:
import os
from getpass import getpass

# Enter your Heroku API Key (hidden input)
HEROKU_API_KEY = getpass("Enter your Heroku API Key: ")
os.environ["HEROKU_API_KEY"] = HEROKU_API_KEY

# Enter your Heroku email
HEROKU_EMAIL = input("Enter your Heroku Email: ")

# 🔧 AUTO-AUTH FIX: Create .netrc file to bypass git login prompts
with open(os.path.expanduser("~/.netrc"), "w") as f:
    f.write(f"machine git.heroku.com login {HEROKU_EMAIL} password {HEROKU_API_KEY}\n")
    f.write(f"machine api.heroku.com login {HEROKU_EMAIL} password {HEROKU_API_KEY}\n")

!chmod 600 ~/.netrc
print("\n✅ Heroku authentication and .netrc configured!")

Enter your Heroku API Key: ··········
Enter your Heroku Email: sxsw8l26@aachendate.de

✅ Heroku authentication and .netrc configured!


## Step 3: Clone Your Repository

In [34]:
# ═══════════════════════════════════════════════════════════════
# ⚠️ EDIT THIS: Your GitHub repository URL
# ═══════════════════════════════════════════════════════════════
GITHUB_REPO = "https://github.com/TELLYHUBCLOUD/NexusMLTB.git"  # ← CHANGE THIS
BRANCH = "main"  # or "master"

# Clone the repo
!rm -rf /content/NexusMLTB
!git clone -b {BRANCH} {GITHUB_REPO} /content/NexusMLTB1
%cd /content/NexusMLTB
print("\n✅ Repository cloned!")

shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
Cloning into '/content/NexusMLTB1'...
fatal: Unable to read current working directory: No such file or directory
[Errno 2] No such file or directory: '/content/NexusMLTB'
/content/NexusMLTB

✅ Repository cloned!


## Step 4: Configure Your Bot

Fill in ALL required values below. Optional ones can be left empty.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 🔧 BOT CONFIGURATION — Fill in your values
# ═══════════════════════════════════════════════════════════════

# ── REQUIRED ──────────────────────────────────────────────────
HEROKU_APP_NAME = ""        # Your Heroku app name (must be unique)
BOT_TOKEN       = ""        # From @BotFather
API_ID          = ""        # From my.telegram.org
API_HASH        = ""        # From my.telegram.org
OWNER_ID        = ""        # Your Telegram user ID
MONGO_URI       = ""        # MongoDB Atlas connection URI

# ── OPTIONAL ─────────────────────────────────────────────────
SUDO_USERS           = ""   # Comma-separated user IDs
FORCE_SUB_CHANNEL_1  = ""   # Channel ID or username
FORCE_SUB_CHANNEL_2  = ""   # Channel ID or username
DB_NAME              = "mirrornexus"
DOWNLOAD_DIR         = "/tmp/downloads"
MAX_SPLIT_SIZE       = "2000000000"
WORKERS              = "500"
STICKER_ID           = ""   # Custom sticker file_id
WELCOME_IMG_API      = ""   # Welcome image API URL
MASSAGE_IMG_URL      = ""   # Fallback welcome image URL
ARIA2C_SECRET        = ""   # Aria2 RPC secret
GDRIVE_FOLDER_ID     = ""   # Google Drive folder ID

print("✅ Configuration set!")

## Step 5: Create Heroku App

In [ ]:
# Choose deployment method: "docker" or "buildpack"
DEPLOY_METHOD = "docker"  # ← "docker" (recommended) or "buildpack"

# Create the Heroku app
!heroku create {HEROKU_APP_NAME} --region us
print(f"\n✅ App '{HEROKU_APP_NAME}' created!")

if DEPLOY_METHOD == "docker":
    !heroku stack:set container -a {HEROKU_APP_NAME}
    print("📦 Stack set to container (Docker)")
else:
    !heroku buildpacks:add --index 1 heroku-community/apt -a {HEROKU_APP_NAME}
    !heroku buildpacks:add --index 2 heroku/python -a {HEROKU_APP_NAME}
    print("📦 Buildpacks configured (apt + python)")

## Step 6: Set Environment Variables on Heroku

In [ ]:
# Set all config vars on Heroku
config_vars = {
    "BOT_TOKEN":            BOT_TOKEN,
    "API_ID":               API_ID,
    "API_HASH":             API_HASH,
    "OWNER_ID":             OWNER_ID,
    "MONGO_URI":            MONGO_URI,
    "HEROKU_APP_NAME":      HEROKU_APP_NAME,
    "DB_NAME":              DB_NAME,
    "DOWNLOAD_DIR":         DOWNLOAD_DIR,
    "MAX_SPLIT_SIZE":       MAX_SPLIT_SIZE,
    "WORKERS":              WORKERS,
}

optional = {
    "SUDO_USERS":           SUDO_USERS,
    "FORCE_SUB_CHANNEL_1":  FORCE_SUB_CHANNEL_1,
    "FORCE_SUB_CHANNEL_2":  FORCE_SUB_CHANNEL_2,
    "STICKER_ID":           STICKER_ID,
    "WELCOME_IMG_API":      WELCOME_IMG_API,
    "MASSAGE_IMG_URL":      MASSAGE_IMG_URL,
    "ARIA2C_SECRET":        ARIA2C_SECRET,
    "GDRIVE_FOLDER_ID":     GDRIVE_FOLDER_ID,
}

for k, v in optional.items():
    if v: config_vars[k] = v

config_string = " ".join(f'{k}="{v}"' for k, v in config_vars.items() if v)
!heroku config:set {config_string} -a {HEROKU_APP_NAME}

print("\n✅ All environment variables configured!")

## Step 7: Deploy to Heroku (With Authentication Fix)

If you get a username/password prompt, it means the fix above failed. But this command uses the API Key directly in the URL to prevent that.

In [10]:
%cd /content/NexusMLTB

HEROKU_APP_NAME = "znexusmltbbot" # ᴢɴᴇxᴜꜱᴍʟᴛʙʙᴏᴛ
# 🔧 AUTO-AUTH FIX: Add remote with credentials embedded
authenticated_url = f"https://:{HEROKU_API_KEY}@git.heroku.com/{HEROKU_APP_NAME}.git"
!git remote remove heroku 2>/dev/null
!git remote add heroku {authenticated_url}

# Configure git identity
!git config user.email "{HEROKU_EMAIL}"
!git config user.name "NexusBot"

# Push to Heroku
print(f"🚀 Pushing to Heroku app: {HEROKU_APP_NAME}...")
!git add -A
!git commit -m "Deploy Mirror Nexus Bot" --allow-empty
!git push heroku {BRANCH}:main -f

print("\n✅ Code pushed to Heroku!")

[Errno 2] No such file or directory: '/content/NexusMLTB'
/content/NexusMLTB
shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
fatal: Unable to read current working directory: No such file or directory
shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
fatal: Unable to read current working directory: No such file or directory
shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
fatal: Unable to read current working directory: No such file or directory
🚀 Pushing to Heroku app: znexusmltbbot...
shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
fatal: Unable to read current working directory: No such file or directory
s

## Step 8: Scale the Worker Dyno

In [ ]:
!heroku ps:scale web=0 worker=1 -a {HEROKU_APP_NAME}
print("\n✅ Worker dyno scaled (web=0, worker=1)!")

## Step 9: Check Status & Logs

In [8]:
!heroku -trail znexusmltbbot

 ›   Warning: -trail is not a heroku command.
 Did you mean drains? (Y/n) Did you mean drains? (Y/n) y Did you mean drains? Yes
 ›   Error: Unexpected argument: znexusmltbbot
 ›   See more help with --help

CLI to interact with Heroku

VERSION
  heroku/11.3.0 linux-x64 node-v22.22.2

USAGE
  $ heroku [COMMAND]

TOPICS
  access          manage user access to apps
  accounts        list the Heroku accounts in your cache
  addons          tools and services for developing, extending, and operating
                  your app
  apps            manage apps on Heroku
  auth            manage authentication for your Heroku account
  authorizations  OAuth authorizations
  buildpacks      scripts used to compile apps
  certs           SSL certificates
  ci              test runner for Heroku Pipelines
  clients         OAuth clients on the platform
  config          environment variables of apps
  container       deploy your Docker-based app to Heroku
  data            list maintenances for an a

In [ ]:
!heroku ps -a {HEROKU_APP_NAME}
print("\n📜 Fetching logs (last 50 lines):")
!heroku logs --tail -n 50 -a {HEROKU_APP_NAME}